# Tune the HAND / REM class thresholds
The plains classes (active floodplain / meander floodplain / bar upland) are absolute thresholds on the
relative elevation model. This notebook shows REM histograms for named test windows so the thresholds in
`config.yaml → hydrology.hand_classes` can be checked against known bars and floodplains.
After editing the config, re-classify without re-routing:

```python
from ecoregions import hydro; hydro.write_hand_classes(cfg)
```

In [ ]:
%matplotlib inline
import numpy as np, matplotlib.pyplot as plt, geopandas as gpd, pandas as pd
from ecoregions.config import load_config, Paths
from ecoregions.grid import Grid, read
from ecoregions.qa import zoom
cfg = load_config("../config.yaml"); paths = Paths(cfg); grid = Grid.load(paths.grid_json)

In [ ]:
import rasterio
from pyproj import Transformer
t = Transformer.from_crs(4326, grid.crs, always_xy=True); inv = ~grid.transform
def window(w, s, e, n):
    xs, ys = t.transform([w, e, w, e], [s, s, n, n]); cr = [inv * (x, y) for x, y in zip(xs, ys)]
    c = [p[0] for p in cr]; r = [p[1] for p in cr]
    return rasterio.windows.Window(int(min(c)), int(min(r)), int(max(c) - min(c)), int(max(r) - min(r)))
tests = {"Sandal Bar (Rechna)": (72.8, 31.2, 73.3, 31.6), "Ganji Bar (Bari)": (73.2, 30.6, 73.7, 30.9),
         "Chenab floodplain": (72.3, 31.1, 72.6, 31.5), "Thal": (71.3, 31.2, 71.8, 31.8)}
fig, ax = plt.subplots(figsize=(9, 5))
for name, bb in tests.items():
    h = rasterio.open(paths.raster("hand")).read(1, window=window(*bb))
    ax.hist(np.clip(h[~np.isnan(h)], -5, 40), bins=90, histtype="step", density=True, label=name)
for k, (lo, hi) in cfg["hydrology"]["hand_classes"].items():
    if lo > -10: ax.axvline(lo, color="k", lw=0.6, ls="--")
ax.set_xlabel("height above nearest major river (m)"); ax.legend();